In [56]:
import pandas as pd
import numpy as np

In [57]:
aligned_frames_df = pd.read_csv('/Users/may/Projects/1p_pipeline/aligned_frames.csv')

/var/folders/k4/zr04khhn74zcj29mf4f0v8hc0000gn/T/ipykernel_31072/1536377271.py:1: DtypeWarning: Columns (22) have mixed types. Specify dtype option on import or set low_memory=False.
  aligned_frames_df = pd.read_csv('/Users/may/Projects/1p_pipeline/aligned_frames.csv')


In [58]:
aligned_frames_df

,global_idx,recorded_idx_neu,Timestamp_neu,is_dropped_neu,Value.ElementType,Value.Size.Width,Value.Size.Height,Value.IsInvalid,Value.Depth,Value.Channels,...,Value.ChannelOfInterest,Value.RegionOfInterest.X,Value.RegionOfInterest.Y,Value.RegionOfInterest.Width,Value.RegionOfInterest.Height,Value.IsClosed,recorded_idx_beh,Timestamp_beh,is_dropped_beh,Value
0,0,0.0,2025-10-28 21:03:43.801088+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,0.0,2025-10-28 21:03:44.056153600+00:00,False,0.0
1,1,1.0,2025-10-28 21:03:43.828211200+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,1.0,2025-10-28 21:03:44.089843200+00:00,False,1.0
2,2,2.0,2025-10-28 21:03:43.860979200+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,2.0,2025-10-28 21:03:44.122854400+00:00,False,2.0
3,3,3.0,2025-10-28 21:03:43.894400+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,3.0,2025-10-28 21:03:44.155545600+00:00,False,3.0
4,4,4.0,2025-10-28 21:03:43.927180800+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,4.0,2025-10-28 21:03:44.188864+00:00,False,4.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54258,54258,54246.0,2025-10-28 21:33:38.586790400+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,NaN,NaN,NaN,NaN
54259,54259,54247.0,2025-10-28 21:33:38.619571200+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,NaN,NaN,NaN,NaN
54260,54260,54248.0,2025-10-28 21:33:38.652518400+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,NaN,NaN,NaN,NaN
54261,54261,54249.0,2025-10-28 21:33:38.685875200+00:00,False,0.0,608.0,608.0,False,U8,1.0,...,0.0,0.0,0.0,608.0,608.0,False,NaN,NaN,NaN,NaN


In [59]:
def detect_missing_indices(df: pd.DataFrame, fps: float, tolerance_s: float, ts_col="Timestamp") -> list[int]:
    expected_dt = 1.0 / fps
    df = df.copy()
    df[ts_col] = pd.to_datetime(df[ts_col], utc=True)
    df = df.sort_values(ts_col).reset_index(drop=True)

    dt = df[ts_col].diff().dt.total_seconds()
    n_dropped_est = np.round(dt / expected_dt - 1).clip(lower=0).astype("Int64")

    drop_events = df[n_dropped_est >= 1]
    missing = []
    for i, k in zip(
        drop_events.index.to_list(),
        n_dropped_est.loc[drop_events.index].astype(int).to_list()
    ):
        missing.extend(range(i + 1, i + 1 + k))

    return missing

In [60]:
def insert_dropped_rows(
    df: pd.DataFrame,
    fps: float,
    missing_idx: list[int],
    ts_col="Timestamp",
):
    df = df.copy()
    df = df.sort_values(ts_col).reset_index(drop=True)
    df["recorded_idx"] = np.arange(len(df), dtype=int)

    expected_dt = pd.to_timedelta(1.0 / fps, unit="s")
    missing_set = set(missing_idx)

    rows = []
    for i in range(len(df)):
        rows.append({
            "global_idx": len(rows),
            "recorded_idx": i,
            ts_col: df.loc[i, ts_col],
            "is_dropped": False,
        })

        if (i + 1) in missing_set:
            k = 0
            j = i + 1
            while j in missing_set:
                k += 1
                j += 1

            last_ts = df.loc[i, ts_col]
            for kk in range(k):
                rows.append({
                    "global_idx": len(rows),
                    "recorded_idx": np.nan,
                    ts_col: last_ts + (kk + 1) * expected_dt,
                    "is_dropped": True,
                })

    filled = pd.DataFrame(rows)

    non_drop = filled["is_dropped"] == False
    filled.loc[non_drop, df.columns.difference([ts_col], sort=False)] = df.loc[
        filled.loc[non_drop, "recorded_idx"].astype(int).values,
        df.columns.difference([ts_col], sort=False)
    ].to_numpy()

    return filled



In [61]:

# ------------------------------------------------------------
# Paths / parameters
# ------------------------------------------------------------

beh_path = r"/Users/may/Projects/1p_pipeline/20251028/beh-cam_frame-id_0.csv"
neu_path = r"/Users/may/Projects/1p_pipeline/20251028/miniscope_frame-id_0.csv"

fps = 30
tolerance_s = 0.002




In [62]:
# ------------------------------------------------------------
# Load behavior + detect drops
# ------------------------------------------------------------

beh = pd.read_csv(beh_path)
beh["Timestamp"] = pd.to_datetime(beh["Timestamp"], utc=True)

beh_missing = detect_missing_indices(
    beh,
    fps=fps,
    tolerance_s=tolerance_s,
    ts_col="Timestamp",
)

beh_filled = insert_dropped_rows(
    beh,
    fps=fps,
    missing_idx=beh_missing,
    ts_col="Timestamp",
)



In [63]:
beh_missing

[2656, 6494, 17248, 18709, 20948, 21323, 27616, 32222]

In [64]:

# ------------------------------------------------------------
# Load neural + detect drops
# ------------------------------------------------------------

neu = pd.read_csv(neu_path)
neu["Timestamp"] = pd.to_datetime(neu["Timestamp"], utc=True)

neu_missing = detect_missing_indices(
    neu,
    fps=fps,
    tolerance_s=tolerance_s,
    ts_col="Timestamp",
)

neu_filled = insert_dropped_rows(
    neu,
    fps=fps,
    missing_idx=neu_missing,
    ts_col="Timestamp",
)


In [65]:
neu_missing

[1199,
 2684,
 3318,
 26262,
 27136,
 27263,
 36373,
 37010,
 41061,
 45975,
 51724,
 52406]

In [66]:
neu_filled

,global_idx,recorded_idx,Timestamp,is_dropped,Value.ElementType,Value.Size.Width,Value.Size.Height,Value.IsInvalid,Value.Depth,Value.Channels,Value.Width,Value.Height,Value.WidthStep,Value.ImageData,Value.ChannelOfInterest,Value.RegionOfInterest.X,Value.RegionOfInterest.Y,Value.RegionOfInterest.Width,Value.RegionOfInterest.Height,Value.IsClosed
0,0,0.0,2025-10-28 21:03:43.801088+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,796913712.0,0.0,0.0,0.0,608.0,608.0,False
1,1,1.0,2025-10-28 21:03:43.828211200+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,540884384.0,0.0,0.0,0.0,608.0,608.0,False
2,2,2.0,2025-10-28 21:03:43.860979200+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,944128048.0,0.0,0.0,0.0,608.0,608.0,False
3,3,3.0,2025-10-28 21:03:43.894400+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,793443504.0,0.0,0.0,0.0,608.0,608.0,False
4,4,4.0,2025-10-28 21:03:43.927180800+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,947069120.0,0.0,0.0,0.0,608.0,608.0,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
54258,54258,54246.0,2025-10-28 21:33:38.586790400+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,984301616.0,0.0,0.0,0.0,608.0,608.0,False
54259,54259,54247.0,2025-10-28 21:33:38.619571200+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,980295728.0,0.0,0.0,0.0,608.0,608.0,False
54260,54260,54248.0,2025-10-28 21:33:38.652518400+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,984301616.0,0.0,0.0,0.0,608.0,608.0,False
54261,54261,54249.0,2025-10-28 21:33:38.685875200+00:00,False,0.0,608.0,608.0,False,U8,1.0,608.0,608.0,608.0,980295728.0,0.0,0.0,0.0,608.0,608.0,False


In [89]:
beh_filled

,global_idx,recorded_idx,Timestamp,is_dropped,Value
0,0,0.0,2025-10-28 21:03:44.056153600+00:00,False,0.0
1,1,1.0,2025-10-28 21:03:44.089843200+00:00,False,1.0
2,2,2.0,2025-10-28 21:03:44.122854400+00:00,False,2.0
3,3,3.0,2025-10-28 21:03:44.155545600+00:00,False,3.0
4,4,4.0,2025-10-28 21:03:44.188864+00:00,False,4.0
...,...,...,...,...,...
53836,53836,53828.0,2025-10-28 21:33:38.578931200+00:00,False,53828.0
53837,53837,53829.0,2025-10-28 21:33:38.612505600+00:00,False,53829.0
53838,53838,53830.0,2025-10-28 21:33:38.645875200+00:00,False,53830.0
53839,53839,53831.0,2025-10-28 21:33:38.679040+00:00,False,53831.0


In [92]:
cols = ["recorded_idx", "Timestamp", "is_dropped"]

neu = neu_filled[cols].copy()
beh = beh_filled[cols].copy()

neu["Timestamp"] = pd.to_datetime(neu["Timestamp"], utc=True)
beh["Timestamp"] = pd.to_datetime(beh["Timestamp"], utc=True)

neu = neu.sort_values("Timestamp").reset_index(drop=True)
beh = beh.sort_values("Timestamp").reset_index(drop=True)

aligned = pd.merge_asof(
    neu,
    beh,
    on="Timestamp",
    direction="nearest",
    tolerance=pd.Timedelta("20ms"),
    suffixes=("_neu", "_beh"),
)


In [104]:
aligned.tail()

,recorded_idx_neu,Timestamp,is_dropped_neu,recorded_idx_beh,is_dropped_beh
54258,54246.0,2025-10-28 21:33:38.586790400+00:00,False,53828.0,False
54259,54247.0,2025-10-28 21:33:38.619571200+00:00,False,53829.0,False
54260,54248.0,2025-10-28 21:33:38.652518400+00:00,False,53830.0,False
54261,54249.0,2025-10-28 21:33:38.685875200+00:00,False,53831.0,False
54262,54250.0,2025-10-28 21:33:38.718515200+00:00,False,53832.0,False


In [105]:
print(len(neu)) 

54263


In [106]:
print(len(beh))

53841


In [107]:
beh_filled[beh_filled["is_dropped"] == True]

,global_idx,recorded_idx,Timestamp,is_dropped,Value
2656,2656,NaN,2025-10-28 21:05:12.620098133+00:00,True,NaN
6495,6495,NaN,2025-10-28 21:07:20.570830933+00:00,True,NaN
17250,17250,NaN,2025-10-28 21:13:19.086658133+00:00,True,NaN
18712,18712,NaN,2025-10-28 21:14:07.792629333+00:00,True,NaN
20952,20952,NaN,2025-10-28 21:15:22.435202133+00:00,True,NaN
21328,21328,NaN,2025-10-28 21:15:34.936053333+00:00,True,NaN
27622,27622,NaN,2025-10-28 21:19:04.739214933+00:00,True,NaN
32229,32229,NaN,2025-10-28 21:21:38.293122133+00:00,True,NaN
